# Notebook 2: Embedding Generation

## Overview
This notebook is the second step in the **Subset Selection Pipeline**. It focuses on generating high-quality embeddings from the formatted text data prepared in Notebook 1.

## Purpose in Subset Selection
Embeddings are vector representations of text that capture semantic meaning. In subset selection, we need embeddings to:
1. **Measure Similarity**: Calculate how similar different data samples are to each other
2. **Enable Facility Location**: The subset selection algorithm uses embeddings to identify diverse, representative samples
3. **Preserve Semantic Information**: Ensure selected subsets maintain the semantic diversity of the original dataset

## What This Notebook Covers
- **Multi-GPU Embedding Generation**: Parallel processing across multiple GPUs for efficient large-scale embedding generation
- **Arctic Encoder Implementation**: Using Snowflake's Arctic embedding model optimized for semantic understanding
- **Batch Processing**: Efficient handling of large datasets through batching and sharding

## Output
- **embeddings.h5**: Merged embedding file containing vector representations of all samples
- Used in Notebook 3 for subset selection via Facility Location algorithm

In [ ]:
# Import from Notebook 1 using %run magic command
# This executes the entire Notebook 1 in the current namespace
%run "data_preparation_and_config.ipynb"

# Additional imports needed for embedding generation
import gc
import logging
import time
from dataclasses import dataclass
from multiprocessing import Pool
from typing import Optional, TypedDict, Dict, List, Union

# Configure logging
logger = logging.getLogger(__name__)
# Third Party Imports (not in Notebook 1)
import h5py
import logging
import os
from tqdm import tqdm
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F
from jinja2 import BaseLoader, Environment

print("✅ Successfully imported from Notebook 1!")
print(f"   • Configuration classes: ProcessingConfig, BasicConfig, EncoderConfig, etc.")
print(f"   • DataProcessor class with utility methods")
print(f"   • config: {type(config).__name__}")
print(f"   • dataset: {len(dataset) if dataset else 'None'} samples")
print(f"   • processor: {type(processor).__name__}")

print("✅ Additional imports for embedding generation loaded!")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

## Arctic Encoder Implementation

### Purpose
Implements the **Snowflake Arctic Embedding Model**, a state-of-the-art encoder specifically designed for generating high-quality semantic embeddings.

### Why Arctic Encoder for Subset Selection?
1. **Semantic Understanding**: Arctic embeddings capture deep semantic meaning, crucial for identifying truly diverse samples
2. **Normalized Embeddings**: Output vectors are L2-normalized, making cosine similarity calculations efficient and meaningful
3. **Long Context**: Supports up to 4096 tokens, handling complex conversations and documents
4. **CLS Pooling**: Uses the [CLS] token embedding as the sentence representation, proven effective for semantic similarity tasks

In [ ]:
# Model configuration 
class ModelConfig(TypedDict):
    pooling_method: str
    normalize_embeddings: bool
    max_length: int
    default_instruction: str
    batch_size: int

@dataclass
class ArcticEncoderConfig:
    model_name: str
    model_config: ModelConfig
    device: torch.device
    num_gpus: int
    batch_size: int
    use_default_instruction: bool
    use_fp16: bool
    testing_mode: bool = False

# Model configurations 
MODEL_CONFIGS: Dict[str, ModelConfig] = {
    "Snowflake/snowflake-arctic-embed-l-v2.0": {
        "pooling_method": "cls",
        "normalize_embeddings": True,
        "max_length": 4096,
        "default_instruction": "Retrieve relevant passages:",
        "batch_size": 24,
    }
}

class ArcticEmbedEncoder:
    """
    Arctic embedding encoder for generating high-quality text embeddings.
    
    Source: arctic_encoder.py - Complete implementation from codebase
    """
    
    def __init__(
        self,
        model_name: str = "Snowflake/snowflake-arctic-embed-l-v2.0",
        device: Optional[torch.device] = None,
        use_fp16: bool = False,
        use_default_instruction: bool = True,
        testing_mode: bool = False,
    ) -> None:
        """Initialize the Arctic encoder."""
        if model_name not in MODEL_CONFIGS:
            raise ValueError(
                f"Model {model_name} not supported. Supported models: {list(MODEL_CONFIGS.keys())}"
            )

        # Use the provided device or default to CUDA
        self.device = device or torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Get device ID for logging
        self.device_id = self.device.index if hasattr(self.device, "index") else 0

        # Configuration
        self.cfg = ArcticEncoderConfig(
            model_name=model_name,
            model_config=MODEL_CONFIGS[model_name],
            device=self.device,
            num_gpus=1,  # Only use 1 GPU per encoder instance
            batch_size=MODEL_CONFIGS[model_name]["batch_size"],
            use_default_instruction=use_default_instruction,
            use_fp16=use_fp16,
            testing_mode=testing_mode,
        )

        self._initialize_model()

    def _initialize_model(self) -> None:
        """Initialize model on the specific GPU."""
        home_dir = os.path.expanduser("~")
        model_path = os.path.join(
            home_dir, ".cache", "instructlab", "models", self.cfg.model_name
        )

        # In testing mode, allow direct download from HuggingFace
        if hasattr(self.cfg, "testing_mode") and self.cfg.testing_mode:
            logger.warning(
                f"Model not found locally at {model_path}. "
                "Testing mode enabled - downloading from HuggingFace..."
            )
            self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_name)
            self.model = AutoModel.from_pretrained(
                self.cfg.model_name,
                add_pooling_layer=False,
                trust_remote_code=True,
            )
        else:
            if not os.path.exists(model_path):
                raise ValueError(
                    f"Model not found in available models: {self.cfg.model_name}\n"
                    "Please run `ilab model download` and download the necessary model"
                )

            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModel.from_pretrained(
                model_path,
                add_pooling_layer=False,
                trust_remote_code=True,
                local_files_only=True,
            )

        if self.cfg.use_fp16:
            self.model = self.model.half()

        self.model = self.model.to(self.cfg.device)
        logger.info(f"Model loaded on device: {self.cfg.device}")

        # Set model to evaluation mode
        self.model.eval()

    def _prepare_inputs(
        self, texts: Union[str, List[str]], instruction: str = ""
    ) -> List[str]:
        """Prepare inputs with model-specific formatting."""
        if isinstance(texts, str):
            texts = [texts]

        # Ensure we always have an instruction
        if not instruction and not self.cfg.use_default_instruction:
            raise ValueError(
                "An instruction must be provided when use_default_instruction is False. "
                "Either provide an instruction or set use_default_instruction to True."
            )

        if (
            not instruction
            and self.cfg.use_default_instruction
            and self.cfg.model_config["default_instruction"]
        ):
            instruction = str(self.cfg.model_config["default_instruction"])

        if not instruction:  # catch if default_instruction is empty
            raise ValueError(
                "No instruction available. Either provide an instruction or ensure "
                "the model config has a valid default_instruction."
            )

        texts = [f"{instruction}: {text}" for text in texts]
        return texts

    @torch.no_grad()
    def encode(
        self,
        inputs: Union[str, List[str]],
        instruction: str = "",
        return_tensors: bool = True,
        show_progress: bool = True,
    ) -> Union[torch.Tensor, np.ndarray]:
        """Encode texts into embeddings."""
        input_was_string = isinstance(inputs, str)
        inputs = self._prepare_inputs(inputs, instruction)

        encodings = self.tokenizer(
            inputs,
            max_length=self.cfg.model_config["max_length"],
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(self.cfg.device)

        embeddings_list = []
        for i in tqdm(
            range(0, len(inputs), self.cfg.batch_size),
            disable=not show_progress or len(inputs) < 256,
            desc=f"Encoding on {self.device}",
        ):
            batch = {k: v[i : i + self.cfg.batch_size] for k, v in encodings.items()}
            outputs = self.model(**batch)
            # Take the first token embedding (CLS) and normalize it
            embeddings = F.normalize(outputs.last_hidden_state[:, 0], p=2, dim=1)
            embeddings_list.append(embeddings.cpu())

        embeddings = torch.cat(embeddings_list, dim=0)
        if input_was_string:
            embeddings = embeddings[0]

        return embeddings if return_tensors else embeddings.numpy()

ENCODER_REGISTRY = {
    "arctic": ArcticEmbedEncoder,
}

def get_encoder_class(encoder_type: str):
    """Get the encoder class based on the encoder type."""
    try:
        if encoder_type not in ENCODER_REGISTRY:
            supported_encoders = list(ENCODER_REGISTRY.keys())
            raise ValueError(
                f"Unsupported encoder type: '{encoder_type}'. "
                f"Supported types are: {supported_encoders}"
            )
        return ENCODER_REGISTRY[encoder_type]
    except Exception as e:
        raise ValueError(f"Error getting encoder class: {str(e)}") from e

print("✅ ArcticEmbedEncoder class and encoder registry defined!")

## Utility Functions for Embedding Generation

These utility functions provide essential infrastructure for robust, scalable embedding generation and subset selection.

In [ ]:
def get_default_num_gpus(testing_mode: bool = False) -> int:
    """
    Get the default number of GPUs based on available CUDA devices.
    
    Source: subset_selection_utils.py
    """
    if not torch.cuda.is_available():
        if testing_mode:
            logger.warning(
                "No CUDA devices detected. Running in testing mode with CPU. "
                "Production use requires GPU acceleration."
            )
            return 1
        raise RuntimeError(
            "No CUDA devices detected. This functionality requires at least one GPU."
        )
    return torch.cuda.device_count()


def compute_pairwise_dense(
    tensor1: torch.Tensor,
    tensor2: Optional[torch.Tensor] = None,
    batch_size: int = 10000,
    metric: str = "cosine",
    device: Optional[Union[str, torch.device]] = None,
    scaling: Optional[str] = None,
    kw: float = 0.1,
) -> torch.Tensor:
    """
    Compute pairwise metric in batches between two sets of vectors.
    
    Source: subset_selection_utils.py
    """
    assert batch_size > 0, "Batch size must be positive."

    if not device:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if tensor2 is None:
        tensor2 = tensor1

    tensor1, tensor2 = tensor1.to(device), tensor2.to(device)
    n_samples1, n_samples2 = tensor1.size(0), tensor2.size(0)
    results = torch.zeros(n_samples1, n_samples2, device="cpu")

    if metric == "cosine":
        tensor1, tensor2 = (
            F.normalize(tensor1, p=2, dim=1),
            F.normalize(tensor2, p=2, dim=1),
        )

    def calculate_metric(a: torch.Tensor, b: torch.Tensor, metric: str, kw: float) -> torch.Tensor:
        if metric in ["cosine", "dot"]:
            return torch.mm(a, b.T)
        if metric == "euclidean":
            distances = torch.cdist(a, b, p=2)
            similarities = 1 / (1 + distances**2)
            return similarities
        if metric == "rbf":
            distance = torch.cdist(a, b)
            squared_distance = distance**2
            avg_dist = torch.mean(squared_distance)
            torch.div(squared_distance, kw * avg_dist, out=squared_distance)
            torch.exp(-squared_distance, out=squared_distance)
            return squared_distance
        raise ValueError(f"Unknown metric: {metric}")

    for i in range(0, n_samples1, batch_size):
        end_i = min(i + batch_size, n_samples1)
        rows = tensor1[i:end_i]

        for j in range(0, n_samples2, batch_size):
            end_j = min(j + batch_size, n_samples2)
            cols = tensor2[j:end_j]
            batch_results = calculate_metric(rows, cols, metric, kw).cpu()
            results[i:end_i, j:end_j] = batch_results

    if scaling == "min-max":
        min_val, max_val = results.min(), results.max()
        if max_val != min_val:
            results = (results - min_val) / (max_val - min_val)
    elif scaling == "additive":
        results = (results + 1) / 2

    return results


def retry_on_exception(max_retries: int = 3, retry_delay: int = 30):
    """
    Decorator to retry a function upon exception up to a maximum number of retries.
    
    Source: subset_selection_utils.py (adapted for standalone use)
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except torch.cuda.OutOfMemoryError as e:
                    last_exception = e
                    logger.error(f"GPU out of memory on attempt {attempt + 1}: {str(e)}")
                except RuntimeError as e:
                    last_exception = e
                    logger.error(f"PyTorch runtime error on attempt {attempt + 1}: {str(e)}")
                except ValueError as e:
                    last_exception = e
                    logger.error(f"Value error on attempt {attempt + 1}: {str(e)}")
                except TypeError as e:
                    last_exception = e
                    logger.error(f"Type error on attempt {attempt + 1}: {str(e)}")
                except IndexError as e:
                    last_exception = e
                    logger.error(f"Index error on attempt {attempt + 1}: {str(e)}")

                if attempt < max_retries - 1:
                    logger.info(f"Retrying in {retry_delay} seconds...")
                    time.sleep(retry_delay)
                    gc.collect()
                    torch.cuda.empty_cache()

            raise last_exception
        return wrapper
    return decorator

print("✅ Utility functions defined!")

# Multi-GPU Embedding Generation Functions (Definition Layer)
Processes a single data shard on a dedicated GPU

In [ ]:
def _process_dataset_shard(args):
    """
    Process a dataset shard on a specific GPU.
    
    Source: subset_selection.py - _process_dataset_shard function
    """
    (
        gpu_id,
        dataset_shard,
        output_dir,
        encoder_type,
        encoder_model,
        instruction,
        template_name,
        templates,
        batch_size,
        testing_mode,
    ) = args

    try:
        # Set the GPU for this process
        if torch.cuda.is_available():
            torch.cuda.set_device(gpu_id)
            device = f"cuda:{gpu_id}"
        else:
            device = "cpu"
            
        logger.info(f"GPU {gpu_id} started processing {len(dataset_shard)} samples")

        encoder_cls = get_encoder_class(encoder_type)
        encoder = encoder_cls(
            model_name=encoder_model,
            device=torch.device(device),
            testing_mode=testing_mode,
        )

        # Set up Jinja environment for templating
        env = Environment(loader=BaseLoader())
        templates_dict = {k: env.from_string(v) for k, v in templates.items()}

        # Create shard-specific output directory
        shard_dir = os.path.join(output_dir, f"shard_{gpu_id}")
        os.makedirs(shard_dir, exist_ok=True)

        # Process batches
        all_embeddings = []
        batch_texts = []

        # Create progress bar
        progress_bar = tqdm(
            desc=f"GPU {gpu_id} generating embeddings",
            total=len(dataset_shard),
            unit=" samples",
            position=gpu_id,  # Stack progress bars
            leave=True,
        )

        # Process each example in the shard
        for example in dataset_shard:
            # Format the text using the template
            template = templates_dict.get(template_name)
            if not template:
                raise ValueError(f"Unknown format type: {template_name}")

            text = template.render(**example)
            batch_texts.append(text)

            # Process when batch is full or at the end
            if len(batch_texts) == batch_size or example == dataset_shard[-1]:
                # Generate embeddings for this batch
                with torch.no_grad():
                    batch_embeddings = (
                        encoder.encode(
                            inputs=batch_texts,
                            instruction=instruction,
                            return_tensors=False,  # Return numpy for easier handling
                        )
                    )

                all_embeddings.append(batch_embeddings)
                progress_bar.update(len(batch_texts))
                batch_texts = []

                # Clean up GPU memory
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        progress_bar.close()

        # Concatenate all batches
        if not all_embeddings:
            logger.warning(f"No embeddings generated for shard on GPU {gpu_id}")
            return None

        embeddings = np.concatenate(all_embeddings, axis=0)

        # Save embeddings to file
        shard_file = os.path.join(shard_dir, f"embeddings_shard_{gpu_id}.h5")
        with h5py.File(shard_file, "w") as h5f:
            h5f.create_dataset("embeddings", data=embeddings, dtype="float32")

        logger.info(f"GPU {gpu_id} completed processing. Saved to {shard_file}")
        return shard_file

    except Exception as e:
        logger.error(f"Error processing shard on GPU {gpu_id}: {str(e)}")
        raise


def _merge_shard_files(shard_files, merged_file):
    """
    Merge all shard files into a single embeddings file.
    
    Source: subset_selection.py - _merge_shard_files function
    """
    logger.info(f"Merging {len(shard_files)} shard files into {merged_file}")

    # Get the shape and type of embeddings from the first shard
    with h5py.File(shard_files[0], "r") as f:
        first_embeddings = f["embeddings"]
        embedding_dim = first_embeddings.shape[1]
        dtype = first_embeddings.dtype

    # Count total samples across all shards
    total_samples = 0
    for shard_file in shard_files:
        with h5py.File(shard_file, "r") as f:
            total_samples += f["embeddings"].shape[0]

    # Create the merged file
    with h5py.File(merged_file, "w") as merged_f:
        merged_dataset = merged_f.create_dataset(
            "embeddings", shape=(total_samples, embedding_dim), dtype=dtype
        )

        # Copy embeddings from each shard
        start_idx = 0
        for shard_file in shard_files:
            with h5py.File(shard_file, "r") as shard_f:
                embeddings = shard_f["embeddings"][:]
                end_idx = start_idx + embeddings.shape[0]
                merged_dataset[start_idx:end_idx] = embeddings
                start_idx = end_idx

            # Remove shard file after merging
            os.remove(shard_file)
            # Remove shard directory if empty
            shard_dir = os.path.dirname(shard_file)
            if not os.listdir(shard_dir):
                os.rmdir(shard_dir)

    logger.info(
        f"Successfully merged embeddings from {len(shard_files)} GPUs with {total_samples} total samples"
    )

print("✅ Multi-GPU embedding generation functions defined!")

# Enhanced DataProcessor for Embedding Generation

In [ ]:
class EmbeddingDataProcessor:
    """
    Enhanced data processor with embedding generation capabilities.
    
    Source: subset_selection.py - DataProcessor.generate_embeddings method
    Extends the DataProcessor from Notebook 1 with embedding generation.
    """
    
    def __init__(self, config: ProcessingConfig):
        """Initialize with configuration from Notebook 1."""
        self.config = config
        self.env = Environment(loader=BaseLoader())
        self.templates = {
            k: self.env.from_string(v) for k, v in config.template.templates.items()
        }
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Set random seeds
        np.random.seed(config.system.seed)
        torch.manual_seed(config.system.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(config.system.seed)

    @retry_on_exception(max_retries=3, retry_delay=30)
    def generate_embeddings(self, dataset, output_dir: str) -> str:
        """
        Generates embeddings for the dataset and saves them to the output directory, using multiple GPUs in parallel.

        Args:
            dataset: The dataset to process.
            output_dir (str): The directory where embeddings will be saved.

        Returns:
            str: The path to the merged embeddings file.
        """
        os.makedirs(output_dir, exist_ok=True)
        merged_path = os.path.join(output_dir, "embeddings.h5")

        # If embeddings already exist, return early
        if os.path.exists(merged_path):
            logger.info(f"Embeddings file already exists in {output_dir}, skipping")
            return merged_path

        # Get number of GPUs to use
        num_gpus = min(self.config.system.num_gpus, torch.cuda.device_count() if torch.cuda.is_available() else 1)
        logger.info(f"Using {num_gpus} GPUs for embedding generation")

        # Create dataset shards - one per GPU
        total_samples = len(dataset)
        per_gpu_samples = (total_samples + num_gpus - 1) // num_gpus  # Ceiling division

        # Prepare arguments for parallel processing
        args_list = []
        for gpu_id in range(num_gpus):
            # Calculate start and end indices for this shard
            start_idx = gpu_id * per_gpu_samples
            end_idx = min(start_idx + per_gpu_samples, total_samples)

            if start_idx >= total_samples:
                continue  # Skip if this GPU has no data to process

            # Create arguments for this GPU
            args_list.append(
                (
                    gpu_id,
                    dataset.select(range(start_idx, end_idx)),
                    output_dir,
                    self.config.encoder.encoder_type,
                    self.config.encoder.encoder_model,
                    self.config.encoder.instruction,
                    self.config.template.template_name,
                    self.config.template.templates,
                    self.config.basic.batch_size,
                    self.config.encoder.testing_mode,
                )
            )

        # Process dataset shards in parallel
        with Pool(processes=num_gpus) as pool:
            shard_files = pool.map(_process_dataset_shard, args_list)

        # Filter out None values (failed shards)
        shard_files = [f for f in shard_files if f is not None]

        if not shard_files:
            raise ValueError("No embeddings were generated from any GPU")

        # Merge all shard files
        _merge_shard_files(shard_files, merged_path)

        return merged_path

print("✅ EmbeddingDataProcessor class defined!")

#  Multi-GPU Embedding Generation (Execution Layer)

In [ ]:
if encoder is not None and 'dataset' in locals():
    print("🎯 Multi-GPU Embedding Generation")
    print("=" * 50)
    
    # Create embedding processor
    embedding_processor = EmbeddingDataProcessor(config)
    
    # Set up output directory
    output_dir = os.path.join(config.basic.output_dir, "embeddings")
    
    print(f"\n📊 Dataset size: {len(dataset):,} samples")
    print(f"💾 Output directory: {output_dir}")
    print(f"🎯 Number of GPUs: {config.system.num_gpus}")
    print(f"📦 Batch size: {config.basic.batch_size}")
    
    # Estimate time based on test encoding
    if 'encode_time' in locals():
        estimated_time = len(dataset) / (len(test_texts) / encode_time)
        print(f"⏱️  Estimated time: {estimated_time / 60:.1f} minutes")
    
    # Generate embeddings
    print(f"\n🚀 Starting multi-GPU embedding generation...")
    start_time = time.time()
    
    try:
        embeddings_file = embedding_processor.generate_embeddings(dataset, output_dir)
        
        generation_time = time.time() - start_time
        
        print(f"\n✅ Embedding generation completed!")
        print(f"⏱️  Total time: {generation_time / 60:.2f} minutes")
        print(f"🎯 Speed: {len(dataset) / generation_time:.2f} samples/sec")
        print(f"💾 Embeddings saved to: {embeddings_file}")
        
        # Show file size
        file_size = os.path.getsize(embeddings_file) / 1024**2
        print(f"📁 File size: {file_size:.1f} MB")
        
    except Exception as e:
        print(f"❌ Error during embedding generation: {e}")
        embeddings_file = None
        
else:
    print("❌ Cannot generate embeddings without encoder and dataset")
    print("Please ensure Notebook 1 has been run and encoder initialization was successful")